In [ ]:
import requests
import pandas as pd
import time
from collections import Counter

BASE_URL = "https://query-api.iedb.org/tcell_search"
INPUT_CSV = "validation_dataset.csv"
OUTPUT_CSV = "main_dataset_with_confidence.csv"
SEQ_COL = "Peptide"
BATCH_SIZE = 50          # peptides per API call (keeps URL length safe)
REQUEST_DELAY = 0.5      # seconds between batches — be polite to the API
DRY_RUN_N = 5            # peptides used for the schema-discovery step

# name substrings used to auto-detect the right columns from the live schema
QUAL_FIELD_HINTS = ["qualitative", "measure", "outcome"]
REF_FIELD_HINTS = ["reference_id", "pubmed", "pmid"]


def fetch_batch(seqs):
    seq_list = ",".join(seqs)
    params = {"linear_sequence": f"in.({seq_list})"}
    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    return r.json()


def main():
    df = pd.read_csv(INPUT_CSV)
    all_seqs = df[SEQ_COL].unique().tolist()
    print(f"{len(all_seqs)} unique peptides to look up.")

    # ---------------- STEP 0: schema discovery ----------------
    print("\n--- Schema discovery (first 5 peptides, all fields) ---")
    sample = fetch_batch(all_seqs[:DRY_RUN_N])
    if not sample:
        print("No records returned for the sample peptides.")
        print("Check connectivity and confirm 'linear_sequence' is still the right")
        print("filter field for tcell_search before proceeding.")
        return

    print("Available fields in tcell_search response:")
    for k in sample[0].keys():
        print(" -", k)

    detected_qual_field, detected_ref_field = None, None
    for k in sample[0].keys():
        lk = k.lower()
        if detected_qual_field is None and any(h in lk for h in QUAL_FIELD_HINTS):
            detected_qual_field = k
        if detected_ref_field is None and any(h in lk for h in REF_FIELD_HINTS):
            detected_ref_field = k

    print(f"\nAuto-detected qualitative-measure field: {detected_qual_field}")
    print(f"Auto-detected reference/evidence-id field: {detected_ref_field}")
    print("^ Verify these against the field list above. If wrong or None,")
    print("  hardcode QUAL_FIELD / REF_FIELD below manually and re-run.")

    # --- MANUAL OVERRIDE: set these explicitly once you've confirmed the schema ---
    QUAL_FIELD = detected_qual_field   # e.g. "qualitative_measure"
    REF_FIELD = detected_ref_field     # e.g. "reference_id"

    if QUAL_FIELD is None:
        print("\nQUAL_FIELD not detected — stopping before the full pull.")
        print("Inspect the printed field list above, set QUAL_FIELD manually, and re-run.")
        return

    # ---------------- STEP 1: full pull ----------------
    print(f"\n--- Full pull: {len(all_seqs)} peptides in batches of {BATCH_SIZE} ---")
    records = []
    for i in range(0, len(all_seqs), BATCH_SIZE):
        batch = all_seqs[i:i + BATCH_SIZE]
        try:
            recs = fetch_batch(batch)
            records.extend(recs)
            print(f"  fetched {min(i + BATCH_SIZE, len(all_seqs))}/{len(all_seqs)} peptides "
                  f"({len(recs)} assay rows this batch)")
        except Exception as e:
            print(f"  batch {i}-{i + len(batch)} failed: {e}")
        time.sleep(REQUEST_DELAY)

    tcell_df = pd.DataFrame(records)
    print(f"\nTotal assay rows retrieved: {len(tcell_df)}")
    if tcell_df.empty:
        print("No assay rows retrieved at all — stopping.")
        return

    # ---------------- STEP 2: aggregate per peptide ----------------
    print("\n--- Aggregating evidence per peptide ---")
    agg_rows = []
    for seq, g in tcell_df.groupby("linear_sequence"):
        qual_counts = Counter(g[QUAL_FIELD].dropna())
        n_evidence = len(g)
        n_refs = g[REF_FIELD].nunique() if REF_FIELD and REF_FIELD in g.columns else None
        majority_label, majority_n = (qual_counts.most_common(1)[0] if qual_counts else (None, 0))
        agreement = majority_n / n_evidence if n_evidence else None
        agg_rows.append({
            SEQ_COL: seq,
            "n_evidence_rows": n_evidence,
            "n_distinct_references": n_refs,
            "qual_measure_counts": dict(qual_counts),
            "majority_qual_label": majority_label,
            "qual_agreement_frac": agreement,
        })
    conf_df = pd.DataFrame(agg_rows)

    # ---------------- STEP 3: merge + confidence flag ----------------
    merged = df.merge(conf_df, on=SEQ_COL, how="left")

    def confidence(row):
        if pd.isna(row.get("n_evidence_rows")):
            return "not_found_in_iedb"
        if row.get("qual_agreement_frac", 1.0) < 1.0:
            return "conflicting"
        if row["n_evidence_rows"] >= 2:
            return "high"
        return "low_single_record"

    merged["negative_confidence"] = merged.apply(confidence, axis=1)

    print("\nConfidence-level distribution (all rows):")
    print(merged["negative_confidence"].value_counts())
    print("\nConfidence-level distribution, label=0 (negative) rows only:")
    print(merged[merged["label"] == 0]["negative_confidence"].value_counts())

    merged.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import ast

INPUT_CSV = "main_dataset_with_confidence.csv"
OUTPUT_CSV = "main_dataset_confidence_refined.csv"
LABEL_COL = "label"

MAJORITY_THRESHOLD = 0.70  # majority label must hold >=70% of evidence to be "high" confidence


def parse_counts(x):
    if isinstance(x, dict):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return {}


def classify_binary(qual_label):
    """Map an IEDB qualitative-measure string to 0/1, handling variants
    like 'Positive-High' / 'Positive-Low' / 'Negative'."""
    if qual_label is None or (isinstance(qual_label, float) and pd.isna(qual_label)):
        return None
    s = str(qual_label).strip().lower()
    if s.startswith("positive"):
        return 1
    if s.startswith("negative"):
        return 0
    return None


def refine(row):
    counts = parse_counts(row.get("qual_measure_counts"))
    if not counts:
        return pd.Series({"confidence_v2": "not_found_in_iedb",
                           "majority_binary": None, "label_mismatch": None})

    total = sum(counts.values())
    majority_label, majority_n = max(counts.items(), key=lambda kv: kv[1])
    frac = majority_n / total if total else 0
    majority_binary = classify_binary(majority_label)

    if total == 1:
        conf = "low_single_record"
    elif frac >= MAJORITY_THRESHOLD:
        conf = "high"
    else:
        conf = "conflicting"

    mismatch = None
    if majority_binary is not None and conf != "conflicting":
        mismatch = (majority_binary != row[LABEL_COL])

    return pd.Series({"confidence_v2": conf,
                       "majority_binary": majority_binary,
                       "label_mismatch": mismatch})


def main():
    df = pd.read_csv(INPUT_CSV)
    refined = df.apply(refine, axis=1)
    df = pd.concat([df, refined], axis=1)

    print(f"Threshold used: majority must hold >= {MAJORITY_THRESHOLD:.0%} of evidence\n")

    print("--- Refined confidence distribution (all rows) ---")
    print(df["confidence_v2"].value_counts())

    print("\n--- Refined confidence, label=0 (negative) only ---")
    print(df[df[LABEL_COL] == 0]["confidence_v2"].value_counts())

    print("\n--- Refined confidence, label=1 (positive) only ---")
    print(df[df[LABEL_COL] == 1]["confidence_v2"].value_counts())

    mismatches = df[df["label_mismatch"] == True]
    print(f"\n--- Label mismatches (dataset label disagrees with IEDB majority evidence) ---")
    print(f"{len(mismatches)} peptides where the assigned label contradicts the majority")
    print("qualitative measure from IEDB (excluding already-'conflicting' cases):")
    if len(mismatches):
        print(mismatches[LABEL_COL].value_counts().rename("count of assigned label among mismatches"))

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved {OUTPUT_CSV}")

    # ---------------- Suggested final keep/drop ----------------
    print("\n--- Suggested final filter ---")
    keep_mask = df["confidence_v2"].isin(["high", "low_single_record"]) & (df["label_mismatch"] != True)
    print(f"Keep (high or low_single_record, no label mismatch): {keep_mask.sum()} / {len(df)}")
    print(df[keep_mask][LABEL_COL].value_counts())


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd

INPUT_CSV = "main_dataset_confidence_refined.csv"
OUTPUT_CSV = "main_dataset_final_clean.csv"
LABEL_COL = "label"

df = pd.read_csv(INPUT_CSV)
print(f"Starting rows: {len(df)}")

# --- drop conflicting ---
before = len(df)
df = df[df["confidence_v2"] != "conflicting"].reset_index(drop=True)
print(f"Dropped {before - len(df)} 'conflicting' rows -> {len(df)} remain")

# --- split mismatches by confidence tier ---
is_mismatch = df["label_mismatch"] == True
is_high = df["confidence_v2"] == "high"
is_low_single = df["confidence_v2"] == "low_single_record"

relabel_mask = is_mismatch & is_high
drop_mask = is_mismatch & is_low_single

print(f"\nRelabeling {relabel_mask.sum()} high-confidence mismatches to majority evidence")
df.loc[relabel_mask, LABEL_COL] = df.loc[relabel_mask, "majority_binary"]

print(f"Dropping {drop_mask.sum()} low-single-record mismatches (not enough evidence to safely relabel)")
df = df[~drop_mask].reset_index(drop=True)

print(f"\nFinal rows: {len(df)}")
print("\nFinal label balance:")
print(df[LABEL_COL].value_counts())

print("\nFinal per-organism balance (sanity check for organism confound):")
print(df.groupby("Organism")[LABEL_COL].agg(["count", "mean"]))

df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {OUTPUT_CSV}")

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import defaultdict

# ---------------- CONFIG ----------------
INPUT_CSV = "main_dataset_final_clean.csv"
OUTPUT_CSV = "validation_dataset_2.csv"
SEQ_COL = "Peptide"
LABEL_COL = "label"
ORG_COL = "Organism"

TINY_ORG_MAX_N = 15            # organism sample-count threshold
TINY_ORG_MIN_PURITY = 0.90     # dominant-class share threshold (1.0 = all one label)
NEARDUP_MAX_HAMMING = 2        # max mismatches between same-length peptides to call "near-duplicate"
MIN_EVIDENCE_ROWS = 2          # only applied if 'n_evidence_rows' column exists
DROP_CONFLICTING_NEARDUPS = True  # drop one side of near-duplicate pairs with conflicting labels

# ---------------- LOAD ----------------
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} peptides. Label balance: {df[LABEL_COL].value_counts().to_dict()}")

# ---------------- STEP 1: drop tiny, near-pure-class organisms ----------------
print("\n--- Step 1: tiny pure-class organisms ---")
org_stats = df.groupby(ORG_COL)[LABEL_COL].agg(["count", "mean"])
org_stats["purity"] = org_stats["mean"].apply(lambda p: max(p, 1 - p))
tiny_pure_orgs = org_stats[
    (org_stats["count"] <= TINY_ORG_MAX_N) & (org_stats["purity"] >= TINY_ORG_MIN_PURITY)
].index.tolist()

if tiny_pure_orgs:
    for org in tiny_pure_orgs:
        n_, p_ = org_stats.loc[org, "count"], org_stats.loc[org, "purity"]
        print(f"  Dropping '{org}': n={int(n_)}, purity={p_:.2f}")
    before = len(df)
    df = df[~df[ORG_COL].isin(tiny_pure_orgs)].reset_index(drop=True)
    print(f"  Removed {before - len(df)} peptides -> {len(df)} remain")
else:
    print("  None found above threshold — nothing dropped.")

# ---------------- STEP 2: evidence-count / confidence filter (guarded) ----------------
print("\n--- Step 2: evidence-count filter ---")
if "n_evidence_rows" in df.columns:
    before = len(df)
    df = df[df["n_evidence_rows"] >= MIN_EVIDENCE_ROWS].reset_index(drop=True)
    print(f"  Kept rows with n_evidence_rows >= {MIN_EVIDENCE_ROWS}: "
          f"removed {before - len(df)} -> {len(df)} remain")
else:
    print("  'n_evidence_rows' column not present in this dataset version — skipped.")
    print("  (If you still have the earlier 1612-row version with this column, merge it in")
    print("   to recover single-evidence-record negatives before trusting them.)")

# ---------------- STEP 3: near-duplicate detection (label-noise + CV-leakage flag) ----------------
print("\n--- Step 3: near-duplicate peptide detection ---")

def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

seqs = df[SEQ_COL].tolist()
labels = df[LABEL_COL].tolist()
n = len(seqs)

parent = list(range(n))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[ry] = rx

by_len = defaultdict(list)
for i, s in enumerate(seqs):
    by_len[len(s)].append(i)

label_conflict_pairs = []
for length, idxs in by_len.items():
    for i, j in combinations(idxs, 2):
        if hamming(seqs[i], seqs[j]) <= NEARDUP_MAX_HAMMING:
            union(i, j)
            if labels[i] != labels[j]:
                label_conflict_pairs.append((i, j, hamming(seqs[i], seqs[j])))

remap, compact_ids = {}, []
for i in range(n):
    r = find(i)
    if r not in remap:
        remap[r] = len(remap)
    compact_ids.append(remap[r])
df["similarity_cluster_id"] = compact_ids

n_clusters = df["similarity_cluster_id"].nunique()
n_multi = (df["similarity_cluster_id"].value_counts() > 1).sum()
print(f"  {n_multi} clusters contain >1 near-duplicate peptide (of {n_clusters} total clusters)")
print(f"  {len(label_conflict_pairs)} near-duplicate pairs have CONFLICTING labels (likely label noise):")
for i, j, d in label_conflict_pairs[:15]:
    print(f"    {seqs[i]} (label={labels[i]})  vs  {seqs[j]} (label={labels[j]})   [hamming={d}]")
if len(label_conflict_pairs) > 15:
    print(f"    ... and {len(label_conflict_pairs) - 15} more (review manually if this list is large)")

if DROP_CONFLICTING_NEARDUPS and label_conflict_pairs:
    drop_idx = {j for _, j, _ in label_conflict_pairs}  # keep first of each pair, drop second
    before = len(df)
    df = df.drop(index=list(drop_idx)).reset_index(drop=True)
    print(f"  Dropped {before - len(df)} peptides on the losing side of label-conflicting pairs -> {len(df)} remain")
    print("  NOTE: this is a simple heuristic (keeps the first-seen peptide). If the conflict list")
    print("  above is long, review it manually instead of trusting the automatic drop.")

# ---------------- STEP 4: organism-based CV grouping column ----------------
print("\n--- Step 4: organism grouping column for validation ---")
df["cv_group"] = df[ORG_COL]
print("  Added 'cv_group' (= Organism) -> pass as groups= in GroupKFold / StratifiedGroupKFold.")
print("  For a stricter group key combining organism + sequence-similarity cluster:")
print("    df['strict_group'] = df['cv_group'].astype(str) + '_' + df['similarity_cluster_id'].astype(str)")

# ---------------- SUMMARY ----------------
print("\n--- Final summary ---")
print(f"Final dataset: {len(df)} peptides")
print(df[LABEL_COL].value_counts())
print("\nPer-organism counts after cleaning:")
print(df.groupby(ORG_COL)[LABEL_COL].agg(["count", "mean"]))

df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved cleaned dataset to {OUTPUT_CSV}")

In [ ]:
! pip install pandas numpy rapidfuzz scikit-learn

In [ ]:
!pip install rapidfuzz --quiet

from google.colab import files
import pandas as pd
from rapidfuzz import fuzz, process

# ---- Config ----
PEPTIDE_COL = "Peptide"
IDENTITY_THRESHOLD = 85.0   # % sequence identity considered a near-duplicate
OUTPUT_NAME = "external_validation_clean.csv"

# ---- 1. Upload files ----
print("Upload external_validation.csv and main_dataset_cleaned_2.csv together:")
uploaded = files.upload()

external_file = None
train_file = None
for fname in uploaded:
    lower = fname.lower()
    if "external" in lower:
        external_file = fname
    elif "main" in lower or "train" in lower or "clean" in lower:
        train_file = fname

if external_file is None or train_file is None:
    raise ValueError(
        f"Could not tell the two files apart from their names: {list(uploaded.keys())}\n"
        f"Rename your external validation file to contain 'external', "
        f"and your training file to contain 'main' or 'train', then re-run this cell."
    )

print(f"External validation file: {external_file}")
print(f"Training file:            {train_file}")

external_df = pd.read_csv(external_file)
train_df = pd.read_csv(train_file)

# ---- 2. Flag near-duplicate peptides -----------------------
train_peptides = train_df[PEPTIDE_COL].tolist()
ext_peptides = external_df[PEPTIDE_COL].tolist()

sim_matrix = process.cdist(ext_peptides, train_peptides, scorer=fuzz.ratio, workers=-1)
best_score = sim_matrix.max(axis=1)
best_idx = sim_matrix.argmax(axis=1)

external_df["max_identity_to_train"] = best_score
external_df["nearest_train_peptide"] = [train_peptides[i] for i in best_idx]

def is_substring_match(pep):
    for tp in train_peptides:
        if pep != tp and (pep in tp or tp in pep):
            return True
    return False

external_df["substring_overlap"] = external_df[PEPTIDE_COL].apply(is_substring_match)
external_df["flagged_overlap"] = (
    external_df["substring_overlap"] | (external_df["max_identity_to_train"] >= IDENTITY_THRESHOLD)
)

n_total = len(external_df)
n_flagged = int(external_df["flagged_overlap"].sum())
print(f"\nFlagged as near-duplicate of a training peptide: {n_flagged} / {n_total} "
      f"({100 * n_flagged / n_total:.1f}%) at identity threshold {IDENTITY_THRESHOLD}%")
print(f"Clean (non-overlapping) peptides remaining: {n_total - n_flagged}")

# ---- 3. Save the clean subset, same columns as the original -----------
clean_df = external_df[~external_df["flagged_overlap"]].drop(
    columns=["max_identity_to_train", "nearest_train_peptide", "substring_overlap", "flagged_overlap"]
)
clean_df.to_csv(OUTPUT_NAME, index=False)
print(f"\nSaved: {OUTPUT_NAME}  ({len(clean_df)} rows, same columns as your original external_validation.csv)")

files.download(OUTPUT_NAME)